# Setting up Python on TACC Vista

Welcome. You're attached to a GH200 compute node through TAP.

This notebook has three acts:

1. **Set up your Python environment** — install uv, build a project, register a Jupyter kernel (all in a terminal).
2. **CPU vs GPU on the brute-force N-body kernel** — feel the speedup.
3. **Collide two Plummer spheres** — watch the merger and tune the parameters.

**Do Act 1 first** — it builds the kernel that Acts 2 & 3 run on. After that, the demos have working defaults; change the numbers at the top of any code cell and re-run to see what happens.


## Act 1 — Set up your Python environment on Vista

Everything in this act happens in a **terminal**, not in this notebook. In JupyterLab: **File → New → Terminal**.

By the end you'll have a `uv`-managed Python project registered as a Jupyter kernel. **Once you finish step 5, switch this notebook's kernel to it (Kernel → Change Kernel) before running any code in Acts 2 & 3.**

Stuck on any step? A pre-built kernel named **`⟨fallback kernel name — TBD⟩`** is already available — pick it from the kernel menu and skip ahead.

### 1. Welcome to Vista

Vista uses **Lmod** to manage software. Get oriented in the terminal:

```bash
module list            # what's loaded right now
module spider cuda     # which CUDA versions are available
module load cuda/13.1  # load the CUDA runtime
module list            # confirm cuda/13.1 is now loaded
nvidia-smi             # see the GH200 GPU and driver
```

**Before continuing, make sure a CUDA runtime is loaded** — `cuda/13.1` should appear in `module list`.

### 2. Install uv

[`uv`](https://docs.astral.sh/uv/) is a fast Python package and project manager. Install it, then put it on your PATH:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
source $HOME/.local/bin/env   # add uv to PATH for this shell
uv --version                  # confirm it works
```

### 3. Create a Python project

```bash
uv init cosmicai-tutorial
cd cosmicai-tutorial
```

`uv init` scaffolds a `pyproject.toml` and pins a Python interpreter for the project. Everything from here runs inside this folder.

### 4. Add dependencies

```bash
uv add torch matplotlib ipykernel tqdm
```

`uv` resolves, downloads, and locks these into a project virtual environment (`.venv/`). NumPy comes along automatically as a dependency of torch and matplotlib.

### 5. Register the environment as a Jupyter kernel

```bash
uv run python -m ipykernel install --user \
    --name cosmicai --display-name "CosmicAI (uv)"
```

This makes the environment selectable inside JupyterLab. Now switch this notebook to it: **Kernel → Change Kernel → "CosmicAI (uv)"** (refresh the kernel list if it doesn't appear yet). You're ready for Act 2.

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device            : {torch.cuda.get_device_name(0)}")
    print(f"Memory            : {props.total_memory / 1e9:.1f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")
    print(f"SMs               : {props.multi_processor_count}")
else:
    print("No CUDA device found — everything below will run on CPU only.")


## Act 2: how fast is your GPU vs your CPU?

We'll evaluate the direct-summation pairwise gravitational force kernel — the heart of every N-body code — on both devices, sweeping $N$ from 2 to 8192 in powers of two.

The force on body $i$ is one vectorized expression with Plummer softening $\epsilon$:

$$\mathbf{a}_i = \sum_{j \ne i} m_j\, \frac{\mathbf{r}_j - \mathbf{r}_i}{\left(|\mathbf{r}_j - \mathbf{r}_i|^2 + \epsilon^2\right)^{3/2}}$$

Expected behavior on the log-log plot: both curves trend toward $\mathcal{O}(N^2)$ at large $N$ (the slope-2 reference is drawn for you). The interesting part is small $N$ — GPU kernel-launch overhead can dominate the actual arithmetic, so the CPU may briefly win. Watch where the curves cross.

In [ ]:
# --- Sweep parameters ---
N_values = [2**k for k in range(1, 14)]   # 2, 4, 8, ..., 8192
n_reps = 5
# ---

import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def gravity_accel(pos, mass, eps=0.05):
    """Plummer-softened pairwise gravitational acceleration. Units: G = 1."""
    dr = pos[None, :, :] - pos[:, None, :]        # (N, N, 3) pairwise
    r2 = (dr * dr).sum(-1) + eps**2               # (N, N) with softening
    inv_r3 = r2.pow(-1.5)
    return (mass[None, :, None] * dr * inv_r3[..., None]).sum(dim=1)

devices = ['cpu'] + (['cuda'] if torch.cuda.is_available() else [])
times = {d: [] for d in devices}

status = display(Markdown("Starting sweep..."), display_id=True)
for j, N in enumerate(N_values):
    status.update(Markdown(f"**Sweeping** N = {N}  ({j+1}/{len(N_values)})"))
    for device in devices:
        pos  = torch.randn(N, 3, device=device)
        mass = torch.ones(N, device=device)
        gravity_accel(pos, mass)                  # warm-up at this N
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_reps):
            gravity_accel(pos, mass)
        if device == 'cuda':
            torch.cuda.synchronize()
        times[device].append((time.perf_counter() - t0) / n_reps)
status.update(Markdown(f"**Done** — swept {len(N_values)} values."))

if 'cuda' in devices:
    print(f"Speedup at N={N_values[-1]}: {times['cpu'][-1] / times['cuda'][-1]:.0f}x")

In [ ]:
%matplotlib inline
# Plot the CPU vs GPU sweep on log-log axes.
fig, ax = plt.subplots(figsize=(7, 5), facecolor='black')
ax.set_facecolor('black')
colors = {'cpu': '#ff7755', 'cuda': '#5599ff'}
for d in devices:
    ax.loglog(N_values, np.array(times[d]) * 1000, '-o',
              color=colors[d], lw=2, label=d.upper())

# O(N^2) reference, anchored to the largest CPU measurement
if 'cpu' in devices:
    ref = times['cpu'][-1] * 1000 * (np.array(N_values, dtype=float) / N_values[-1])**2
    ax.loglog(N_values, ref, '--', color='gray', alpha=0.5, label=r'$\mathcal{O}(N^2)$')

ax.set_xlabel('N (bodies)')
ax.set_ylabel('Time per force eval (ms)')
ax.set_ylim(bottom=1e-2)
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
for spine in ax.spines.values():
    spine.set_color('white')
ax.legend(facecolor='black', labelcolor='white', edgecolor='white')
ax.grid(True, which='both', alpha=0.2)
plt.show()

## Act 3: collide two Plummer spheres

Now the real demo. We sample two Plummer spheres, point them at each other, and let leapfrog do its thing.

**Knobs:**

- `N_per_cluster` — particles per cluster (memory and time scale as $N^2$)
- `impact_b` — impact parameter in cluster scale lengths. `0.0` = head-on, larger = glancing
- `v_approach` — approach speed as a fraction of the mutual escape velocity at first contact. `<1` → bound merger, `>1` → flyby
- `mass_ratio` — $M_2/M_1$. `1.0` = major merger, smaller = minor merger with tidal stripping

The default values give an equal-mass, slightly off-axis, bound merger. Solid baseline. Now break it.

> CPU note: the defaults assume you're on the GPU. If you ended up on a CPU-only node, drop `N_per_cluster` to ~512 and `n_steps` to ~100 or you'll be here a while.

In [ ]:
# --- Tune these ---
N_per_cluster  = 4096   # particles per cluster
impact_b       = 0.5    # impact parameter (cluster scale lengths)
v_approach     = 0.8    # fraction of escape velocity (<1 bound, >1 flyby)
mass_ratio     = 1.0    # M2 / M1
n_steps        = 1500   # leapfrog steps
dt             = 0.02   # step size (1.0 ≈ one dynamical time)
snapshot_every = 15     # save every Nth step for the animation
# ---

In [ ]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def sample_plummer(N, total_mass, scale_a, device):
    """Sample N particles from a Plummer sphere.
    Positions ~ Plummer density profile.
    Velocities ~ isotropic Gaussian at the local 1D velocity dispersion.
    Units: G = 1."""
    u = torch.rand(N, device=device)
    r = scale_a / torch.sqrt(u.pow(-2/3) - 1)
    direction = torch.randn(N, 3, device=device)
    direction = direction / direction.norm(dim=1, keepdim=True)
    pos = r[:, None] * direction
    sigma = torch.sqrt(total_mass / (6 * torch.sqrt(r**2 + scale_a**2)))
    vel = torch.randn(N, 3, device=device) * sigma[:, None]
    return pos, vel

M1, M2 = 1.0, mass_ratio
sep0 = 8.0                                   # initial center-to-center separation
v_esc_pair = (2 * (M1 + M2) / sep0) ** 0.5
v_rel = v_approach * v_esc_pair

# Heavier cluster gets a larger scale length so densities stay comparable.
a2 = mass_ratio ** (1/3)
pos1, vel1 = sample_plummer(N_per_cluster, M1, 1.0, device)
pos2, vel2 = sample_plummer(N_per_cluster, M2, a2,  device)

pos1 = pos1 + torch.tensor([-sep0/2, +impact_b/2, 0.0], device=device)
pos2 = pos2 + torch.tensor([+sep0/2, -impact_b/2, 0.0], device=device)
vel1 = vel1 + torch.tensor([+v_rel/2, 0.0, 0.0], device=device)
vel2 = vel2 + torch.tensor([-v_rel/2, 0.0, 0.0], device=device)

pos = torch.cat([pos1, pos2], dim=0)
vel = torch.cat([vel1, vel2], dim=0)
mass = torch.cat([
    torch.full((N_per_cluster,), M1 / N_per_cluster, device=device),
    torch.full((N_per_cluster,), M2 / N_per_cluster, device=device),
])

# --- Leapfrog (kick-drift-kick) ---
snapshots = []
t0 = time.perf_counter()
acc = gravity_accel(pos, mass)
status = display(Markdown("Starting leapfrog..."), display_id=True)
milestone = max(1, n_steps // 20)            # update display every 5%
for step in range(n_steps):
    if step % milestone == 0:
        status.update(Markdown(
            f"**Leapfrog** step {step:>5}/{n_steps}  ({100 * step // n_steps}%)"
        ))
    vel = vel + 0.5 * dt * acc
    pos = pos + dt * vel
    acc = gravity_accel(pos, mass)
    vel = vel + 0.5 * dt * acc
    if step % snapshot_every == 0:
        snapshots.append(pos.detach().cpu().numpy().copy())
if device == 'cuda':
    torch.cuda.synchronize()
status.update(Markdown(f"**Leapfrog** complete — {n_steps} steps."))

print(f"Simulated {n_steps} steps x {2*N_per_cluster} bodies in "
      f"{time.perf_counter() - t0:.1f}s on {device.upper()}.")
print(f"Saved {len(snapshots)} snapshots for the animation.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['animation.embed_limit'] = 1024  # MB; default 20 is tight for 8k-point scatter
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, Markdown
import uuid

snaps = np.array(snapshots)   # (n_snap, N_total, 3)

fig = plt.figure(figsize=(12, 6), facecolor='black')

# --- 2D xy projection (left) ---
ax2 = fig.add_subplot(1, 2, 1)
ax2.set_facecolor('black')
ax2.set_xlim(-8, 8); ax2.set_ylim(-8, 8); ax2.set_aspect('equal')
ax2.set_xlabel('x'); ax2.set_ylabel('y')
ax2.tick_params(colors='white')
ax2.xaxis.label.set_color('white'); ax2.yaxis.label.set_color('white')
for spine in ax2.spines.values():
    spine.set_color('white')

scat2a = ax2.scatter([], [], s=1, c='#5599ff', alpha=0.6)
scat2b = ax2.scatter([], [], s=1, c='#ff7755', alpha=0.6)

# --- 3D scatter (right), with a slow auto-rotating camera ---
ax3 = fig.add_subplot(1, 2, 2, projection='3d')
ax3.set_facecolor('black')
ax3.set_xlim(-8, 8); ax3.set_ylim(-8, 8); ax3.set_zlim(-8, 8)
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('z')
ax3.tick_params(colors='white')
for axis in (ax3.xaxis, ax3.yaxis, ax3.zaxis):
    axis.label.set_color('white')
    axis.pane.fill = False
    axis.pane.set_edgecolor('white')

scat3a = ax3.scatter([], [], [], s=1, c='#5599ff', alpha=0.6)
scat3b = ax3.scatter([], [], [], s=1, c='#ff7755', alpha=0.6)

title = fig.suptitle("", color='white')
n_frames = len(snaps)

def update(i):
    s = snaps[i]
    scat2a.set_offsets(s[:N_per_cluster, :2])
    scat2b.set_offsets(s[N_per_cluster:, :2])
    scat3a._offsets3d = (s[:N_per_cluster, 0],
                         s[:N_per_cluster, 1],
                         s[:N_per_cluster, 2])
    scat3b._offsets3d = (s[N_per_cluster:, 0],
                         s[N_per_cluster:, 1],
                         s[N_per_cluster:, 2])
    ax3.view_init(elev=20, azim=30 + i * 0.5)   # ~0.5 deg per frame
    title.set_text(f"t = {i * snapshot_every * dt:.2f} dynamical times")
    return scat2a, scat2b, scat3a, scat3b, title

ani = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

# CSS spinner: keeps spinning while the kernel is blocked in to_jshtml().
spinner_html = """
<style>
  @keyframes _demo_spin { to { transform: rotate(360deg); } }
  ._demo_spinner {
    display: inline-block; width: 14px; height: 14px;
    border: 2px solid rgba(127,127,127,0.3);
    border-top-color: #5599ff;
    border-radius: 50%;
    animation: _demo_spin 0.8s linear infinite;
    vertical-align: middle; margin-right: 8px;
  }
</style>
<div><span class="_demo_spinner"></span>Rendering animation...</div>
"""
status = display(HTML(spinner_html), display_id=True)
html = ani.to_jshtml(default_mode='reflect')
status.update(Markdown(f"Animation ready — {n_frames} frames."))

# Wrap in a unique-id container and click Play once the widget is in the DOM.
anim_id = f"anim_{uuid.uuid4().hex[:8]}"
autoplay_js = (
    f'<div id="{anim_id}">{html}</div>'
    '<script>'
    f'(function() {{ var tries = 0; var t = setInterval(function() {{ '
    f'var div = document.getElementById("{anim_id}"); '
    'if (div) { var btns = div.querySelectorAll(\'button[title="Play"]\'); '
    'if (btns.length > 0) { btns[0].click(); clearInterval(t); return; } } '
    'if (++tries > 20) clearInterval(t); '
    '}, 100); })();'
    '</script>'
)
display(HTML(autoplay_js))

## Try these

Re-run the simulation cell with these tweaks and watch what changes:

- **Push the speed past escape.** Set `v_approach = 1.5`. Does it still merge?
- **Head-on collision.** Set `impact_b = 0.0`. Bullet through bullet — does the system retain any angular momentum?
- **Glancing blow.** Set `impact_b = 2.0`. What's the threshold where the clusters fly past instead of merging?
- **Minor merger.** Set `mass_ratio = 0.1`. Watch the small cluster get tidally shredded.
- **Cheap and ugly.** Drop `N_per_cluster = 256`. Does the merger still *look* right? Where does the picture break down?

If you push `N_per_cluster` to ~16000 you're approaching the limits of direct summation — beyond that, real cosmological codes switch to tree (Barnes-Hut) or particle-mesh methods. That's a story for a different lecture.
